# 13.3 Reference Counting and Garbage Collection

**Prerequisites:** 13.1 Objects, Names and the Heap, 13.2 The Call Stack and Frames, 6.3 Context Managers  
**Target:** Python 3.12+ (notes flag 3.13/3.14 differences)

### What you'll learn
- Why CPython frees most objects **immediately**, and what that buys you
- 🔴 The one case refcounting cannot handle: **reference cycles**
- The generational collector — what `gc.collect()` actually does
- 🔴 Why `__del__` is not a destructor and should almost never be written
- `weakref` and `WeakValueDictionary` — caches that cannot leak
- Finding a real leak: a registry with no eviction policy
- ⚠️ Every number here is CPython-specific; PyPy has no reference counting at all

---

## Two mechanisms, not one

CPython frees memory in two quite different ways, and confusing them is the source of most
myths about "Python's garbage collector".

| Mechanism | Handles | When it runs |
|---|---|---|
| **Reference counting** | the overwhelming majority of objects | 🔴 the instant the last reference goes |
| **The cyclic collector** (`gc`) | only objects trapped in **reference cycles** | periodically, by allocation pressure |

**13.1** showed the counting half: a task record went 2 → 3 → 4 → 3 → 2 as containers took and
released references. When that count reaches zero, the object is freed *right there* — no
pause, no scheduler, no waiting.

That property is genuinely valuable and unusual. A file, a socket or a database connection is
released at a predictable moment rather than "eventually".

## Deterministic release

`__del__` runs when the last reference disappears. Used purely as an observation tool it shows
the timing precisely.

In [ ]:
import gc
import sys
import weakref


class Connection:
    """Stands in for a real pooled database connection."""

    def __init__(self, dsn):
        self.dsn = dsn

    def __del__(self):
        print(f"      -> released {self.dsn}")


print("open two connections:")
primary = Connection("postgres://db-1/orders")
replica = Connection("postgres://db-2/orders")

print("\ndrop the replica:")
replica = None
print("   (freed above, before this line ran)")

print("\nrebind primary to something else:")
primary = "no longer a connection"
print("   (same story - rebinding dropped the last reference)")

print("\nnothing was scheduled or deferred; both went at a known instant.")

Each connection is released **between the two `print` calls that
surround it** — not at the end of the cell, not at some later collection. Rebinding a name has
exactly the same effect as `del`, because what matters is the count, not the syntax.

This is why context managers (**6.3**) and CPython's refcounting feel similar in practice, and
why plenty of Python code gets away without `with`. 🔴 **It is still not a substitute.**
Refcount timing is a CPython implementation detail; PyPy frees the same object at an
unpredictable later moment, and an exception mid-function can keep a frame — and therefore its
locals — alive far longer than you expect (**13.2**). Use `with` for anything that must be
closed.

## 🔴 The gap: reference cycles

Reference counting has exactly one blind spot. If two objects reference each other, each keeps
the other's count above zero — even when nothing else in the program can reach either of them.

This is not exotic. It is a doubly linked list (**14.4**), a tree node with a parent pointer
(**14.7**), a cached object holding the cache, or any two service objects that know about each
other.

In [ ]:
class Worker:
    """A worker that keeps a pointer back to its supervisor, and vice versa."""

    def __init__(self, name):
        self.name = name
        self.peer = None

    def __del__(self):
        print(f"      -> collected {self.name}")


def build_pair():
    """Both objects become unreachable when this returns - but they reference each other."""
    supervisor = Worker("supervisor")
    worker = Worker("worker-1")
    supervisor.peer = worker
    worker.peer = supervisor
    # no return: nothing outside this function can reach either object


gc.collect()                       # start from a clean slate
gc.disable()                       # switch OFF the cycle collector
try:
    build_pair()
    print("after build_pair() returned:")
    print("   nothing was printed above, so BOTH objects are still alive")
    print("   refcounting alone cannot free them\n")

    print("now run the cycle collector by hand:")
    unreachable = gc.collect()
    print(f"   gc.collect() reclaimed {unreachable} unreachable objects")
finally:
    gc.enable()                    # always restore it
    print("\ncycle collector re-enabled:", gc.isenabled())

The pair survives the function returning. Neither count reaches zero
because each object holds the other, so the refcounting half of the system is helpless.

`gc.collect()` finds them and frees both — note that the `collected ...` lines appear
**before** the count is printed, because the collector runs the finalisers and only then
returns how many objects it reclaimed.

🔴 **You do not normally call `gc.collect()`.** It runs automatically. The cell disabled it
deliberately so the gap was visible; in real code the cycle would be cleaned up within
milliseconds. What matters is knowing the mechanism exists, because it explains why memory can
be released *later* than refcounting alone would suggest.

## Generations

The collector is **generational**, on the observation that most objects die young. New objects
start in generation 0, and anything that survives a collection is promoted.

In [ ]:
thresholds = gc.get_threshold()
counts = gc.get_count()

print("generation thresholds :", thresholds)
print("current counts        :", counts)
print()
print("  gen 0 is collected once (allocations - deallocations) exceeds the")
print(f"  first threshold ({thresholds[0]}). Survivors are promoted to gen 1,")
print("  then gen 2. Older generations are collected far less often, which is")
print("  what keeps the collector cheap.")
print()
print("per-generation statistics since interpreter start:")
for index, stats in enumerate(gc.get_stats()):
    print(f"  gen {index}: collections={stats['collections']:>6,}  "
          f"collected={stats['collected']:>8,}  uncollectable={stats['uncollectable']}")

print("\nobjects the collector is currently tracking:", f"{len(gc.get_objects()):,}")
print("  (only container types are tracked - an int or a str cannot form a cycle)")

The thresholds printed above are **not** the `(700, 10, 10)` quoted in
most older material — CPython has retuned them, and 3.13's incremental collector reworked how
the oldest generation is handled. Read the values from `gc.get_threshold()` rather than
trusting any figure you remember.

`uncollectable` is the number worth watching: it counts objects the collector *found* but could
not free. On a healthy modern interpreter it stays at zero.

Note the last line: only **container** types are tracked. An `int` or a `str` cannot reference
anything, so it can never be part of a cycle and the collector ignores it entirely.

## 🔴 `__del__` is not a destructor

This notebook uses `__del__` as an observation tool because it is the only way to *see* the
timing. That is nearly the only good reason to write one.

| Problem | What happens |
|---|---|
| Timing is not guaranteed | on PyPy it runs much later; at interpreter shutdown it may not run at all |
| Exceptions vanish | an exception inside `__del__` cannot propagate — it is printed and ignored |
| It can resurrect the object | storing `self` somewhere during `__del__` brings it back |
| It runs on a partly-built object | if `__init__` raised halfway, `__del__` still runs |

The exception behaviour is the one that actually bites, because a cleanup failure disappears
silently instead of failing your tests.

In [ ]:
class LeakyHandler:
    def __init__(self, name):
        self.name = name

    def __del__(self):
        raise RuntimeError(f"cleanup of {self.name} failed")


print("dropping an object whose __del__ raises:")
handler = LeakyHandler("metrics-flusher")
handler = None
print("   ...and execution carried on as if nothing happened.")
print()
print("   The traceback went to stderr and was swallowed. Nothing was raised,")
print("   nothing can catch it, and no test will fail because of it.")
print()
print("   Use a context manager (6.3) or an explicit .close() instead:")
print("   with connect(dsn) as conn:   <- deterministic, catchable, testable")

The `RuntimeError` never reached the caller. Python printed
`Exception ignored ...` to stderr and continued. In a service that log line is easy to miss,
and the resource you thought you were cleaning up simply was not.

🔴 **Rule of thumb: if cleanup matters, make it explicit.** `with` (**6.3**) gives you a
guaranteed, catchable, testable release point. `__del__` is a last-resort safety net at best,
and `weakref.finalize` is a better one when you genuinely need it.

## `weakref` — referring without owning

A **weak** reference lets you point at an object without keeping it alive. That single property
solves the most common memory leak in long-running services: a cache or registry that
accumulates for ever because nothing ever removes entries.

In [ ]:
class Metric:
    """A per-service metrics bucket - the kind of object a registry accumulates."""

    def __init__(self, name):
        self.name = name
        self.samples = [0.0] * 50


def metrics_alive():
    return sum(1 for obj in gc.get_objects() if isinstance(obj, Metric))


# ---- the leak: a plain dict that nothing ever evicts from ----
REGISTRY = {}
gc.collect()
print("the leak - a registry with no eviction:")
print("   Metric objects alive at start :", metrics_alive())

for i in range(5_000):
    REGISTRY[f"svc-{i}"] = Metric(f"svc-{i}")
print("   after registering 5,000       :", metrics_alive())

REGISTRY.clear()
gc.collect()
print("   after REGISTRY.clear()        :", metrics_alive())
print("   ^ they were only ever alive because the dict held them")

# ---- the fix: hold them weakly ----
print("\nthe fix - a weak cache alongside the real owner:")
CACHE = weakref.WeakValueDictionary()
owned = [Metric(f"svc-{i}") for i in range(5_000)]      # the genuine owner
for metric in owned:
    CACHE[metric.name] = metric

print("   cache entries while owned     :", len(CACHE))

owned.clear()
gc.collect()
print("   cache entries after owners go :", len(CACHE))

del metric                                              # the loop variable!
gc.collect()
print("   after dropping the loop name  :", len(CACHE))

The plain dict is the entire bug: 5,000 `Metric` objects stay alive
purely because `REGISTRY` references them, and clearing it drops every one. In a real service
nobody ever clears it, and the process grows until it is restarted.

The weak cache behaves completely differently. While `owned` holds the metrics the cache is
full; the moment the real owner lets go, the entries **evict themselves**. No eviction policy,
no TTL, no bookkeeping.

🔴 Look closely at the second-to-last line: **one entry survives** the owners being dropped.
That is not a flaw in `weakref` — it is the loop variable `metric`, still bound to the final
`Metric` from the `for` loop. It is a real reference, so the object is genuinely still alive and
the cache is right to keep it. Dropping that name takes the cache to zero.

This is **13.1**'s lesson arriving with consequences: *a name is a reference*, and a stray one
is indistinguishable from a deliberate one.

### Choosing a weak container

| Type | Weak in | Use for |
|---|---|---|
| `weakref.ref(obj)` | the single referent | a back-pointer that must not own its target |
| `WeakValueDictionary` | the **values** | 🔴 caches keyed by id where the owner is elsewhere |
| `WeakKeyDictionary` | the **keys** | attaching metadata to objects you do not own |
| `WeakSet` | the members | observer/listener registries |
| `weakref.finalize` | — | cleanup that is safer than `__del__` |

⚠️ Not everything is weak-referenceable. `int`, `str`, `tuple`, `list` and `dict` are not,
because they have no slot to store the weak reference in. Your own classes are — unless they
define `__slots__` without including `"__weakref__"` (**13.4**).

---

## Common Mistakes & Pitfalls

1. 🔴 **Calling `gc.collect()` in normal code.** It is a debugging tool. Reaching for it in production usually means hiding a reference you should have released.
2. 🔴 **Writing `__del__` for cleanup.** Exceptions in it vanish, timing is not guaranteed, and it may never run at shutdown. Use `with` (**6.3**) or `weakref.finalize`.
3. **Assuming refcount timing is part of Python.** It is CPython behaviour; PyPy frees later and in a different order.
4. **Believing the cyclic collector handles everything.** It handles cycles. Everything else is refcounting, and a live reference defeats both.
5. 🔴 **A cache or registry with no eviction.** The single most common leak in a long-running service. Bound it, or hold it weakly.
6. **Forgetting the loop variable.** After `for x in items:`, `x` still references the last item — as the cell above demonstrates.
7. **Holding a traceback.** `exc.__traceback__` keeps every frame, and therefore every local, alive (**13.2**).
8. **Expecting `__slots__` classes to be weak-referenceable.** Add `"__weakref__"` to the slots if you need it (**13.4**).
9. **Quoting `(700, 10, 10)` as the thresholds.** Measure them; they have changed.

## Best Practices

- Use `with` for anything holding a real resource. Do not rely on refcount timing.
- Bound every cache: a size limit, a TTL, or `WeakValueDictionary`.
- Prefer `weakref.finalize` over `__del__` when cleanup genuinely cannot be explicit.
- Break cycles deliberately with a weak back-pointer — parent links in trees are the classic case (**14.7**).
- Use `functools.lru_cache(maxsize=...)` rather than an unbounded dict; the default `None` means unbounded.
- Reach for `tracemalloc` (**17.5**) to find *where* memory is allocated; use `gc` to understand *why* it is retained.
- Do not store exception objects long-term without clearing `__traceback__` first.
- Treat `gc.get_stats()['uncollectable'] > 0` as a bug worth investigating.

## Practice Exercises

Try these before moving on.

1. Reproduce the cycle demo, then break it by making one side a `weakref.ref`. Confirm both objects die without `gc.collect()`.
2. 🔴 Take the leaking `REGISTRY` and fix it three ways: a size bound, a TTL, and `WeakValueDictionary`. Which is right for a metrics registry, and why?
3. Write a tree node with a parent pointer (**14.7**). Show it forms a cycle, then fix it with `weakref.ref` and prove the fix.
4. Add `__slots__` to `Metric` without `"__weakref__"`. What breaks, and what is the error?
5. Catch an exception, keep it in a module-level list, and show with `gc.get_objects()` that its frames are still alive (**13.2**).
6. Measure `gc.get_stats()` before and after allocating a million short-lived objects. Which generation did the work?
7. 🔴 Disable the collector in a long-running loop that creates cycles and watch memory grow with `tracemalloc` (**17.5**). Re-enable and watch it recover.
8. **Interview question:** *“Python has reference counting, so why does it need a garbage collector at all?”* Answer with a concrete cycle.

---

## Version notes

| Version | Change |
|---|---|
| **3.4** | PEP 442 — objects with `__del__` in a cycle became collectable; the old `gc.garbage` graveyard is effectively history |
| **3.12** | 🔴 PEP 683 immortal objects — `None`, `True` and small ints are never refcounted or freed (**13.1**) |
| **3.13** | 🔴 An **incremental** cyclic collector, which changes how the oldest generation is scanned and cuts long pauses |
| **3.13+** | The free-threaded build changes refcounting substantially (biased/deferred counting); `gc` semantics are preserved but timing is not (**12.1**) |
| — | Generation thresholds have been retuned; read `gc.get_threshold()` rather than quoting a remembered figure |

> 🔴 **Reference counting is not part of Python.** It is how CPython happens to work. PyPy uses
> a tracing collector with no counts at all, so `__del__` timing, refcount tricks and
> "the file closes when it goes out of scope" are all CPython-only. Code that must be portable
> uses `with`.

## 13 How Python Works Under the Hood — the folder

| Notebook | Covers |
|---|---|
| **13.1** | objects on the heap, names, identity, interning, immortality |
| **13.2** | the call stack, frames, recursion limits, tracebacks |
| **13.3** | this notebook — reference counting, cycles, the collector, `weakref` |
| **13.4** | where the memory actually goes — `getsizeof`, container overhead, `__slots__` measured |
| **13.5** | attribute lookup and class creation — descriptors, `__init_subclass__`, metaclasses |

**The one-sentence version:** *refcounting frees almost everything the instant it becomes
unreachable, the cyclic collector exists solely for the cases it cannot, and a leak in Python
is almost always a reference you forgot you were holding.*

## Related

- **6.3** Context Managers — the deterministic cleanup you should be using
- **13.1** Objects, Names and the Heap — where the counting starts
- **13.2** The Call Stack and Frames — tracebacks and suspended frames as retention sources
- **14.4 / 14.7** Linked Lists, Trees — where cycles arise naturally
- **17.5** Profiling and Performance — `tracemalloc`, for *where* the memory went